# Livrable Éthique — Projet IA for HumanForYou

**Formation :** PGE A4 FISA INFO — Intelligence Artificielle (Machine Learning)  
**Projet :** Analyse de l'attrition des employés — HumanForYou  
**Données :** HR Analytics — Kaggle (vjchoudhary7)


## Introduction

Ce document présente la démarche éthique adoptée tout au long du projet d'analyse de l'attrition des employés de l'entreprise pharmaceutique HumanForYou. Conformément aux **7 exigences recommandées par la Commission Européenne** pour une IA digne de confiance, nous avons questionné chacun de nos choix méthodologiques — de la sélection des données jusqu'aux recommandations finales — afin de garantir une approche responsable, transparente et respectueuse des individus.

L'objectif du projet est d'identifier les facteurs influençant le départ des employés et de proposer un modèle prédictif permettant à HumanForYou de mettre en place des actions de rétention ciblées. Cette finalité, bien qu'à visée organisationnelle, touche directement aux conditions de travail et à la vie professionnelle de personnes réelles, ce qui impose une **vigilance éthique particulière**.


## 1. Respect de l'autonomie humaine

### Principe
Le modèle d'IA ne doit pas se substituer à la décision humaine. Les employés concernés doivent rester acteurs de leur trajectoire professionnelle.

### Application au projet

**Ce que fait notre modèle :** Il attribue à chaque employé une probabilité de départ estimée sur la base de ses données professionnelles et personnelles. Il ne décide pas — il informe.

**Décision d'équipe :** Nous avons clairement positionné notre modèle comme un **outil d'aide à la décision RH**, et non comme un système de décision automatisé. Les recommandations produites doivent être soumises à l'interprétation et au jugement d'un responsable RH ou d'un manager avant toute action.

**Point de vigilance :** Le risque principal est que les résultats du modèle soient utilisés de manière automatique (ex : refus de promotion, surveillance accrue d'un employé identifié "à risque") sans intervention humaine. Nous recommandons à HumanForYou d'intégrer une charte d'utilisation précisant que toute décision individuelle doit être validée par un humain.

**Variables concernées :**
- `PerformanceRating`, `JobInvolvement` : issues de l'évaluation managériale — elles reflètent une appréciation subjective. Leur poids dans le modèle doit être interprété avec prudence pour éviter de pénaliser un employé sur la base d'un jugement partial.
- `avg_hours_worked` : calculée à partir des badgeuses — un employé parti tôt un jour pour raison médicale ne doit pas être systématiquement considéré comme "moins impliqué".


## 2. Robustesse technique et sécurité

### Principe
Le modèle doit être fiable, stable, et ses erreurs doivent être anticipées et minimisées.

### Application au projet

**Gestion du déséquilibre des classes :** Le jeu de données présente 84% d'employés restants contre 16% de départs. Sans correction, un modèle naïf prédirait systématiquement "reste" et afficherait 84% d'accuracy sans aucune valeur prédictive réelle. Nous avons utilisé le paramètre `class_weight='balanced'` pour corriger ce biais et prioriser la détection des vrais départs.

**Métriques choisies :** Nous avons privilégié l'AUC-ROC et le recall plutôt que l'accuracy seule, car dans ce contexte, **manquer un employé sur le départ (faux négatif) est plus coûteux** que de déclencher une action de rétention inutile (faux positif).

**Validation du modèle :** Un train/test split stratifié (80/20) a été appliqué pour évaluer les performances sur des données non vues. La stratification garantit que le ratio 84/16 est conservé dans les deux sous-ensembles, assurant une évaluation représentative.

**Point de vigilance :** Le modèle a été entraîné sur des données de 2015. Des changements organisationnels, économiques ou sectoriels depuis cette date peuvent rendre le modèle moins pertinent. Un **réentraînement périodique** est recommandé.

**Variables concernées :**
- `EnvironmentSatisfaction`, `JobSatisfaction`, `WorkLifeBalance` : issues d'une enquête **non obligatoire** — les valeurs manquantes ont été imputées par la médiane. Cette imputation introduit une approximation qui doit être signalée : les employés n'ayant pas répondu peuvent avoir des profils spécifiques (ex : désengagement).
- `avg_hours_worked` : calculée sur une année complète de badgeuse. Les jours sans données ont été exclus du calcul pour ne pas biaiser la moyenne.


## 3. Confidentialité et gouvernance des données

### Principe
Les données personnelles doivent être protégées, leur usage limité à la finalité annoncée, et leur traitement conforme aux réglementations en vigueur (RGPD).

### Application au projet

**Anonymisation :** Les données fournies par HumanForYou sont **anonymisées** — chaque employé est représenté uniquement par son `EmployeeID`, sans nom ni information directement identifiante. Nous avons supprimé cette colonne avant toute modélisation car elle n'a aucune valeur prédictive.

**Données sensibles identifiées et traitées :**

| Variable | Nature sensible | Décision d'équipe |
|----------|----------------|-------------------|
| `Gender` | Donnée à caractère personnel — sexe | Conservée pour analyse de biais, surveillée |
| `Age` | Donnée personnelle | Conservée (facteur légitime en RH) |
| `MaritalStatus` | Vie privée | Conservée avec vigilance (voir section 5) |
| `MonthlyIncome` | Donnée financière sensible | Conservée — facteur majeur d'attrition |

**Variables supprimées :**
- `EmployeeID` : identifiant direct — supprimé avant modélisation
- `EmployeeCount` : constante (valeur 1 pour tous) — aucune information prédictive
- `StandardHours` : constante (valeur 8 pour tous) — aucune information prédictive
- `Over18` : constante (True pour tous) — aucune information prédictive

**Décision d'équipe :** Les données des badgeuses (`in_time`, `out_time`) constituent une **surveillance des horaires de travail**. Leur utilisation a été jugée légitime dans ce contexte car elles permettent de calculer une variable agrégée (`avg_hours_worked`) et non de tracer individuellement les employés heure par heure.

**Point de vigilance :** Dans un contexte réel, l'utilisation de données de badgeuse à des fins prédictives RH devrait faire l'objet d'une **information préalable des employés** et d'une consultation des représentants du personnel, conformément au RGPD (articles 13 et 14).


## 4. Transparence

### Principe
Le fonctionnement du modèle doit être explicable. Les parties prenantes doivent comprendre comment les décisions sont prises.

### Application au projet

**Choix du modèle :** Nous avons choisi la **régression logistique** comme modèle de référence, notamment pour son **interprétabilité** : les coefficients permettent de quantifier l'effet de chaque variable sur la probabilité de départ. Contrairement à des modèles de type boîte noire (deep learning), la régression logistique permet d'expliquer à un responsable RH pourquoi un employé est identifié comme à risque.

**Interprétation des coefficients :** Nous avons extrait et visualisé les 15 variables les plus influentes du modèle. Chaque variable est présentée avec son effet directionnel (augmente ou réduit le risque de départ).

**Documentation du notebook :** Chaque étape est documentée avec des cellules markdown expliquant les choix méthodologiques, les tests statistiques utilisés et leur interprétation.

**Point de vigilance :** Si un modèle plus complexe (Random Forest, XGBoost) est retenu pour ses meilleures performances, il faudra s'appuyer sur des outils d'explicabilité comme les **feature importances** pour maintenir la transparence vis-à-vis des utilisateurs finaux.


## 5. Diversité, non-discrimination et équité

### Principe
Le modèle ne doit pas reproduire ni amplifier des discriminations liées au genre, à l'âge, au statut marital ou à tout autre critère protégé.

### Application au projet

C'est l'exigence la plus critique pour ce projet car plusieurs variables du dataset sont des **critères potentiellement discriminatoires** au sens du droit du travail.

**`Gender` (Sexe)**  
- *Risque :* un modèle entraîné sur des données historiques peut reproduire des inégalités existantes (ex : si les femmes quittent davantage l'entreprise à cause d'un manque d'équité salariale, le modèle "apprend" que le genre prédit le départ).
- *Décision :* conservée pour l'analyse exploratoire afin d'identifier d'éventuelles inégalités, mais son utilisation comme critère de décision individuelle est exclue.
- *Point de vigilance :* **utiliser le genre comme critère de ciblage RH est illégal** en France (article L1132-1 du Code du travail).

**`Age` (Âge)**  
- *Risque :* discrimination par l'âge (seniors identifiés comme "moins fidèles").
- *Décision :* conservé comme facteur légitime d'analyse démographique. Les résultats doivent être présentés en termes de politiques de fidélisation par tranche d'âge, non de scoring individuel.

**`MaritalStatus` (Statut marital)**  
- *Risque :* les célibataires montrent un taux d'attrition plus élevé dans nos données. Cibler ces profils serait discriminatoire.
- *Décision :* conservé pour l'analyse bivariée uniquement. Recommandation explicite à HumanForYou de **ne jamais prendre de décision individuelle sur la base du statut marital**.

**`MonthlyIncome` (Salaire)**  
- *Risque faible* : variable légitime et centrale dans l'analyse de l'attrition.
- *Décision :* conservée sans restriction — c'est un levier d'action direct pour l'entreprise.

**Recommandation :** Vérifier lors du déploiement que les taux de faux positifs/négatifs sont comparables entre groupes (hommes/femmes, jeunes/seniors). Un écart significatif indiquerait un biais systémique à corriger.


## 6. Bien-être environnemental et sociétal

### Principe
Le développement et l'utilisation de l'IA doivent limiter leur impact environnemental et contribuer positivement à la société.

### Application au projet

**Impact environnemental :**  
Les modèles utilisés (régression logistique, Random Forest) sont des algorithmes classiques peu gourmands en ressources computationnelles. Leur empreinte carbone est négligeable comparée à l'entraînement de modèles de deep learning à grande échelle. Le projet a été développé localement, sans recours à des infrastructures cloud massives.

**Impact sociétal positif :**  
En aidant HumanForYou à réduire son taux d'attrition, le projet contribue à améliorer les **conditions de travail** des employés si les recommandations portent sur des leviers comme la satisfaction au travail, l'équilibre vie pro/perso ou les opportunités de formation. La réduction du turnover bénéficie également aux équipes en place : moins de surcharge liée aux départs, meilleure continuité des projets.

**Risque sociétal identifié :**  
Un usage mal encadré du modèle pourrait conduire à une **surveillance accrue** des employés ou à des pratiques de gestion par la peur. Cela aurait un effet délétère sur le bien-être au travail, à l'opposé de l'objectif initial.

**Décision d'équipe :** Nos recommandations finales se concentrent sur des **actions positives** (amélioration des conditions, formations, politique salariale) et non sur des mesures coercitives ou de surveillance individuelle.


## 7. Responsabilité

### Principe
Les responsabilités doivent être clairement définies. En cas d'erreur du modèle, des mécanismes de correction doivent exister.

### Application au projet

**Responsabilité de l'équipe projet :**  
Nous sommes responsables de la qualité de l'analyse, de la pertinence des modèles choisis et de la clarté des recommandations. Toute limitation identifiée (imputations, déséquilibre de classes, données datées de 2015) est documentée dans le notebook et dans ce livrable.

**Responsabilité de HumanForYou :**  
L'entreprise est responsable de l'usage qu'elle fait des résultats. Nous recommandons :
- Une **charte d'utilisation** précisant les usages autorisés et interdits
- Un **comité de revue** impliquant RH, managers et représentants des employés avant toute action basée sur les résultats
- Un **droit de recours** pour tout employé s'estimant lésé par une décision informée par le modèle

**Traçabilité :**  
Le notebook Jupyter constitue la trace complète et reproductible de notre démarche. Les choix d'exclusion de variables sont documentés et justifiés. Les hyperparamètres des modèles sont explicitement définis dans le code.

**Mécanismes de correction :**  
- Révision annuelle du modèle avec de nouvelles données
- Réévaluation régulière des métriques pour détecter une dérive (data drift)
- Processus de signalement des erreurs permettant aux RH de remonter les cas où le modèle s'est trompé


## Synthèse des points de vigilance

| Exigence | Point de vigilance principal | Action recommandée |
|----------|-----------------------------|-----------------|
| Autonomie humaine | Risque d'automatisation des décisions RH | Charte d'utilisation obligatoire |
| Robustesse | Données datées de 2015 | Réentraînement annuel |
| Confidentialité | Données de badgeuse et données sensibles | Information préalable des employés |
| Transparence | Modèles complexes peu interprétables | Privilégier les modèles explicables |
| Non-discrimination | Gender, Age, MaritalStatus dans le modèle | Ne jamais cibler individuellement sur ces critères |
| Bien-être sociétal | Risque de surveillance accrue | Recommandations orientées actions positives |
| Responsabilité | Absence de traçabilité des décisions | Comité de revue + droit de recours |

## Conclusion

La démarche éthique adoptée dans ce projet s'est construite progressivement, depuis le choix des variables jusqu'à la formulation des recommandations. Plusieurs décisions d'équipe ont été motivées par des préoccupations éthiques : la suppression des identifiants directs, la vigilance sur les variables potentiellement discriminatoires, le choix de modèles interprétables, et le positionnement systématique de notre outil comme aide à la décision humaine et non comme système autonome.

Le cas HumanForYou illustre une tension fondamentale de l'IA appliquée aux ressources humaines : les données les plus prédictives (genre, âge, statut marital) sont souvent celles qui présentent le plus grand risque éthique. La réponse à cette tension n'est pas de supprimer ces variables aveuglément, mais d'en encadrer strictement l'usage et de former les utilisateurs finaux à une lecture critique des résultats.
